Sources
https://aboskovic21.github.io/projects/pu_gb.pdf
optuna

In [68]:


import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)
mpl.rcParams['text.usetex'] = False

In [69]:
# Main code voor processen van de data
import uproot
import sklearn
import xgboost
import pathlib
import matplotlib.pyplot as plt
import scienceplots
import numpy as np
from matplotlib.pyplot import figure
import corner

figure(figsize=(8, 6), dpi=80)

# Background
with uproot.open(pathlib.Path(r"Training_Data\data.root")) as file:
    tree = file['treeMLDplus']
    tree.show()
    branches = tree.keys()
    print(branches)
    training_data_bkg = tree.arrays(branches, library="np")
    training_data_bkg['species'] = np.zeros(len(training_data_bkg['inv_mass']))
    # data_matrix = np.column_stack([data[b][:10000] for b in branches])

    # plt.figure(dpi=300)
    # fig = corner.corner(data_matrix, labels=branches)
    # inv_mass_training = tree.arrays(['inv_mass'],library='np')

# Signal
with uproot.open(pathlib.Path(r"Training_Data\FD.root")) as file:
    tree = file['treeMLDplus']
    tree.show()
    branches = tree.keys()
    print(branches)
    training_data_FD = tree.arrays(branches, library="np")
    training_data_FD['species'] = np.ones(len(training_data_FD['inv_mass']))
    # data_matrix = np.column_stack([data[b][:10000] for b in branches])

    # plt.figure(dpi=300)
    # fig = corner.corner(data_matrix, labels=branches)
    # inv_mass_FD = tree.arrays(['inv_mass'],library='np')


name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
inv_mass             | float                    | AsDtype('>f4')
pt_cand              | float                    | AsDtype('>f4')
d_len                | float                    | AsDtype('>f4')
d_len_xy             | float                    | AsDtype('>f4')
norm_dl_xy           | float                    | AsDtype('>f4')
cos_p                | float                    | AsDtype('>f4')
cos_p_xy             | float                    | AsDtype('>f4')
imp_par_xy           | float                    | AsDtype('>f4')
sig_vert             | float                    | AsDtype('>f4')
max_norm_d0d0exp     | float                    | AsDtype('>f4')
nsigComb_Pi_0        | float                    | AsDtype('>f4')
nsigComb_K_0         | float                    | AsDtype('>f4')
nsigComb_Pi_1        | float                    | AsDtype(

In [70]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

kbg_df = pd.DataFrame(training_data_FD)
FD_df = pd.DataFrame(training_data_bkg)

full_data = pd.concat([kbg_df,FD_df])


In [ ]:
x_data = full_data[branches]
y_data = full_data['species']
print(set(y_data))


x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.2, random_state=7)
print(np.unique(y_train, return_counts=True))

# Settings we train the xgbclassifier with, the options we pass to the xgbclassifier are the hyperparameters we want to optimise.
pos_class_weight = len(training_data_bkg['inv_mass']) / len(training_data_FD['inv_mass'])
model = xgb.XGBClassifier(
    n_estimators=100,
    objective='binary:logistic',
    scale_pos_weight=pos_class_weight,
    max_delta_step=1,
    random_state=42
)
model.fit(x_train, y_train)

{0.0, 1.0}
(array([0., 1.]), array([3741792,    4560]))


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
from sklearn.metrics import recall_score, precision_score, roc_auc_score, accuracy_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
# Define a function to evaluate the results
def evaluate_results(y_test, y_predict):
    print('Classification results:')
    f1 = f1_score(y_test, y_predict)
    print("f1: %.2f%%" % (f1 * 100.0)) 
    roc = roc_auc_score(y_test, y_predict)
    print("roc: %.2f%%" % (roc * 100.0)) 
    rec = recall_score(y_test, y_predict, average='binary')
    print("recall: %.2f%%" % (rec * 100.0)) 
    prc = precision_score(y_test, y_predict, average='binary')
    print("precision: %.2f%%" % (prc * 100.0))

# Evaluate the model
predictions = model.predict(x_test)
print("Confusion Matrix:")
print(confusion_matrix(y_test, predictions))
print("\nClassification Report:")
print(classification_report(y_test, predictions))

# Make predictions on the testing set and evaluate the results
y_predict = model.predict(x_test)
evaluate_results(y_test, y_predict)

Confusion Matrix:
[[928095   7281]
 [    84   1128]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      0.99      1.00    935376
         1.0       0.13      0.93      0.23      1212

    accuracy                           0.99    936588
   macro avg       0.57      0.96      0.62    936588
weighted avg       1.00      0.99      1.00    936588

Classification results:
f1: 23.45%
roc: 96.15%
recall: 93.07%
precision: 13.41%


In [74]:
import optuna
# Define an objective function. This will find how good the model is performing, based on different hyperparameters.
# Using this function optuna will sample the hyperparameterspace to find the best hyperparameters for training.
def objective(trial):
    data, target =  full_data[branches], full_data['species']
    train_x, test_x, train_y, test_y = train_test_split(data, target, test_size=0.25)
    dtrain = xgb.DMatrix(train_x, label=train_y)
    dtest = xgb.DMatrix(test_x, label=test_y)

    param = {
        "silent": 1,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "booster": trial.suggest_categorical("booster", ["gbtree", "gblinear", "dart"]),
        "lambda": trial.suggest_loguniform("lambda", 1e-8, 1.0),
        "alpha": trial.suggest_loguniform("alpha", 1e-8, 1.0),
    }

    if param["booster"] == "gbtree" or param["booster"] == "dart":
        param["max_depth"] = trial.suggest_int("max_depth", 1, 9)
        param["eta"] = trial.suggest_loguniform("eta", 1e-8, 1.0)
        param["gamma"] = trial.suggest_loguniform("gamma", 1e-8, 1.0)
        param["grow_policy"] = trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"])
    if param["booster"] == "dart":
        param["sample_type"] = trial.suggest_categorical("sample_type", ["uniform", "weighted"])
        param["normalize_type"] = trial.suggest_categorical("normalize_type", ["tree", "forest"])
        param["rate_drop"] = trial.suggest_loguniform("rate_drop", 1e-8, 1.0)
        param["skip_drop"] = trial.suggest_loguniform("skip_drop", 1e-8, 1.0)

    # Add a callback for pruning.
    # We are sampling the hyperparameterspace for what combination of hyperparameters results in a fast learning algorithm
    # For samples that learn very slowly, we do not need to train them for many epochs
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "validation-auc")
    bst = xgb.train(param, dtrain, evals=[(dtest, "validation")], callbacks=[pruning_callback])
    preds = bst.predict(dtest)
    pred_labels = np.rint(preds)
    accuracy = sklearn.metrics.accuracy_score(test_y, pred_labels)
    return accuracy

study = optuna.create_study()
study.optimize(objective, n_trials=100)
print(study.best_trial)

[I 2025-11-19 18:17:23,258] A new study created in memory with name: no-name-1b2b3017-8526-4c87-b280-2f8e579867b4
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:15: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "lambda": trial.suggest_loguniform("lambda", 1e-8, 1.0),
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "alpha": trial.suggest_loguniform("alpha", 1e-8, 1.0),
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:21: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v

[0]	validation-auc:0.94931
[1]	validation-auc:0.94987
[2]	validation-auc:0.94987
[3]	validation-auc:0.94987
[4]	validation-auc:0.95001
[5]	validation-auc:0.95001
[6]	validation-auc:0.95001
[7]	validation-auc:0.95001
[8]	validation-auc:0.94998
[9]	validation-auc:0.94997


[I 2025-11-19 18:17:30,777] Trial 0 finished with value: 0.9987879409089162 and parameters: {'booster': 'gbtree', 'lambda': 0.4810716996124017, 'alpha': 9.28924761961769e-08, 'max_depth': 5, 'eta': 4.560840933869212e-05, 'gamma': 0.3504592858575302, 'grow_policy': 'lossguide'}. Best is trial 0 with value: 0.9987879409089162.
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:15: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "lambda": trial.suggest_loguniform("lambda", 1e-8, 1.0),
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "alpha": trial.suggest_loguniform("alpha", 1e-8, 1.0),
c:\Users\sa

[0]	validation-auc:0.94770
[1]	validation-auc:0.95235
[2]	validation-auc:0.95194
[3]	validation-auc:0.95014
[4]	validation-auc:0.94852
[5]	validation-auc:0.94727
[6]	validation-auc:0.94601
[7]	validation-auc:0.94495
[8]	validation-auc:0.94400
[9]	validation-auc:0.94319


[I 2025-11-19 18:17:38,548] Trial 1 finished with value: 0.9986615245977954 and parameters: {'booster': 'gblinear', 'lambda': 0.003379645707605756, 'alpha': 1.1297613117536401e-05}. Best is trial 1 with value: 0.9986615245977954.
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:15: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "lambda": trial.suggest_loguniform("lambda", 1e-8, 1.0),
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "alpha": trial.suggest_loguniform("alpha", 1e-8, 1.0),
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:21: FutureWarning: suggest_loguniform has be

...

[0]	validation-auc:0.50000
[1]	validation-auc:0.50000
[2]	validation-auc:0.50000
[3]	validation-auc:0.50000
[4]	validation-auc:0.50000
[5]	validation-auc:0.50000
[6]	validation-auc:0.50000
[7]	validation-auc:0.50000
[8]	validation-auc:0.50000
[9]	validation-auc:0.50000


[I 2025-11-19 18:27:43,323] Trial 97 finished with value: 0.99875121184555 and parameters: {'booster': 'gblinear', 'lambda': 3.390212213718534e-05, 'alpha': 0.14518921824576236}. Best is trial 1 with value: 0.9986615245977954.
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:15: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "lambda": trial.suggest_loguniform("lambda", 1e-8, 1.0),
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:16: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  "alpha": trial.suggest_loguniform("alpha", 1e-8, 1.0),
C:\Users\sande\AppData\Local\Temp\ipykernel_22444\2102006868.py:21: FutureWarning: suggest_loguniform has been 

[0]	validation-auc:0.50000
[1]	validation-auc:0.50000
[2]	validation-auc:0.50000
[3]	validation-auc:0.50000
[4]	validation-auc:0.50000
[5]	validation-auc:0.50000
[6]	validation-auc:0.50000
[7]	validation-auc:0.50000
[8]	validation-auc:0.50000
[9]	validation-auc:0.50000


[I 2025-11-19 18:28:07,920] Trial 99 finished with value: 0.9987580451596647 and parameters: {'booster': 'gblinear', 'lambda': 6.359799862376515e-05, 'alpha': 0.9742463078398288}. Best is trial 1 with value: 0.9986615245977954.


FrozenTrial(number=1, state=1, values=[0.9986615245977954], datetime_start=datetime.datetime(2025, 11, 19, 18, 17, 30, 778870), datetime_complete=datetime.datetime(2025, 11, 19, 18, 17, 38, 548313), params={'booster': 'gblinear', 'lambda': 0.003379645707605756, 'alpha': 1.1297613117536401e-05}, user_attrs={}, system_attrs={}, intermediate_values={0: 0.9477020133539393, 1: 0.9523488150631018, 2: 0.9519440213542262, 3: 0.950137036459109, 4: 0.9485228599732272, 5: 0.9472702783224912, 6: 0.946009100807637, 7: 0.9449458393977115, 8: 0.9440032125359619, 9: 0.9431914281518381}, distributions={'booster': CategoricalDistribution(choices=('gbtree', 'gblinear', 'dart')), 'lambda': FloatDistribution(high=1.0, log=True, low=1e-08, step=None), 'alpha': FloatDistribution(high=1.0, log=True, low=1e-08, step=None)}, trial_id=1, value=None)


In [ ]:
import plotly

optuna.visualization.plot_param_importances(study)


ImportError: Tried to import 'plotly' but failed. Please make sure that the package is installed correctly to use this feature. Actual error: No module named 'plotly'.